# Atom-Versatz im zeitunabhaengigen Profil

Das **Atom** wandert aus der Mitte nach aussen, das Lichtfeld steht still.
Mit ihm wandern alle Regionen, ueber die ausgewertet wird. Gerechnet wird mit
`lib/atom_offset.py` - hier steht nur, was eingetragen, gezeichnet und
gespeichert wird.

Ergebnis: ein PDF und eine CSV in `Notebook_Ergebnisse/`.

In [1]:
# ====================  PARAMETER  ====================================
WAIST_UM     = 1.04      # Waist in der Atomebene (um)
WIDTH_MHZ    = 0.37      # Tonabstand; die Spannweite d des 3x4-Arrays folgt daraus
R_X, R_Y     = 0.97, 1.16   # Amplitudenverhaeltnisse
PROFIL       = "airy"    # "airy" oder "gauss"

ATOM_T_UK    = 17.0      # Atomtemperatur (uK)  -> sigma_atom
NU_R_KHZ     = 60.4      # radiale Fallenfrequenz (kHz)

RICHTUNGEN   = ("x", "y", "xy")   # "x" horizontal, "y" vertikal, "xy" diagonal
                                  # eine Kurve je Richtung; ("x",) = nur horizontal
R_MAX_ANTEIL = 0.5       # 0.5 = bis an den aeusseren Ton (Rand des Tonarrays)
N_PUNKTE     = 41        # Stuetzstellen von r = 0 bis r_max
HART         = False     # True: harte und kombinierte Groessen zusaetzlich (doppelte Rechenzeit)
LOG_Y        = False     # log-Achse; mit HART = True noetig, sonst verschwindet die gewichtete Kurve

NAME         = "AtomOffset_Richtungen"   # Dateiname-Kern
# =====================================================================

import sys
from pathlib import Path
BASE = Path.cwd()                 # Ordner Combinated_Optimization
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
OUT = BASE / "Notebook_Ergebnisse"
OUT.mkdir(exist_ok=True)
print("Ausgabe nach:", OUT)

Ausgabe nach: /mnt/user-data/uploads/Multitone_Simulation/Amplitudes/Combinated_Optimization/Notebook_Ergebnisse


In [2]:
# ====================  RECHNEN  ======================================
import numpy as np
from lib import atom_offset as ao

params = dict(ao.DEFAULTS,
              waist=WAIST_UM * 1e-6, width=WIDTH_MHZ * 1e6,
              r_x=R_X, r_y=R_Y, profile=PROFIL,
              atom_temperature=ATOM_T_UK * 1e-6, trap_freq_r=NU_R_KHZ * 1e3,
              hart=HART)

opt   = ao.make_optimizer(params)
d_um  = ao.width_um(params, opt)          # Spannweite des Tonarrays
rand_um = R_MAX_ANTEIL * d_um             # halbe Spannweite = aeusserer Ton
print(f"Spannweite Tonarray d = {d_um:.4f} um   ->   Rand bei {rand_um:.4f} um")
print(f"sigma_atom = {opt.sigma_atom*1e9:.1f} nm,   pitch = {opt.pitch*1e6:.3f} um")

# Jede Richtung laeuft bis zu IHREM Rand: horizontal und vertikal bis zum
# aeusseren Ton (d/2), die Diagonale erst am Eckton (d/2 * sqrt(2)).
R_FAKTOR = {"x": 1.0, "y": 1.0, "xy": np.sqrt(2.0), "antixy": np.sqrt(2.0)}


def zeige(i, g):
    if i % 10 == 0 or i == g:
        print(f"  {i}/{g}")
    return True


res = {}
for rid in RICHTUNGEN:
    ende = rand_um * R_FAKTOR[rid]
    print(f"Richtung {rid}: bis r = {ende:.4f} um")
    res[rid] = ao.sweep(params, ende * 1e-6, n=N_PUNKTE, richtungen=(rid,),
                        progress=zeige)
print("fertig")

Spannweite Tonarray d = 2.3370 um   ->   Rand bei 1.1685 um
sigma_atom = 107.6 nm,   pitch = 5.288 um
Richtung x: bis r = 1.1685 um


  10/41


  20/41


  30/41


  40/41


...

  10/41


  20/41


  30/41


  40/41


  41/41
fertig


In [3]:
# ====================  PLOT  =========================================
import matplotlib.pyplot as plt

FIG_W_IN, PT = 16.0 / 2.54, 10.0          # 16 cm = \textwidth, 10 pt Fliesstext
# (Schluessel, Index am Symbol, Klartext, Linienart)
FAMILIEN = [("hart",     "h", "hard",     "--"),
            ("weighted", "w", "weighted", "-"),
            ("kombi",    "c", "combined", ":")]
if not HART:
    FAMILIEN = [f for f in FAMILIEN if f[0] == "weighted"]
# Farbe = Richtung, Linienart = Familie
FARBEN = {"x": "#0072B2", "y": "#D55E00", "xy": "#009E73", "antixy": "#CC79A7"}
TEXTE = {"x": "Hor.", "y": "Vert.", "xy": "Diag.", "antixy": "Antidiag."}

mehrere = len(FAMILIEN) > 1 or len(RICHTUNGEN) > 1
r_ende = max(res[rid]["offset_um"][-1] for rid in RICHTUNGEN)
fig, (ax_u, ax_c) = plt.subplots(1, 2, figsize=(FIG_W_IN, 0.42 * FIG_W_IN))
for ax, groesse, sym in ((ax_u, "uniformity", "U"), (ax_c, "crosstalk", r"\eta")):
    for rid in RICHTUNGEN:
        r_um = res[rid]["offset_um"]
        for fam, idx, fname, ls in FAMILIEN:
            y = res[rid][f"{groesse}_{fam}__{rid}"] * 100.0
            name = TEXTE[rid] if len(FAMILIEN) == 1 else f"{TEXTE[rid]}, {fname}"
            ax.plot(r_um, y, color=FARBEN[rid], ls=ls, lw=1.4, label=name)
    # Der aeussere Ton der Achsenrichtungen - die Diagonale laeuft weiter.
    if r_ende > rand_um * 1.01:
        ax.axvline(rand_um, color="0.55", ls=":", lw=1.0, zorder=0)
    ax.set_xlabel(r"$r$ ($\mathrm{\mu}$m)")
    # Nur eine Familie -> der Index steht am Symbol (U_w), sonst in der Legende.
    ax.set_ylabel(rf"${sym}$ (%)" if len(FAMILIEN) > 1
                  else rf"${sym}_{FAMILIEN[0][1]}$ (%)")
    ax.set_xlim(0.0, r_ende)
    if LOG_Y:
        ax.set_yscale("log")
    ax.grid(alpha=0.25, lw=0.5)
    # zweite Achse: Versatz in Einheiten der halben Spannweite (1 = aeusserer Ton)
    sek = ax.secondary_xaxis("top", functions=(lambda v: v / rand_um,
                                               lambda v: v * rand_um))
    sek.set_xlabel(r"$r\,/\,(d/2)$")
    # Legende in BEIDE Plots, oben links - aber EINZEILIG und mit Luft nach
    # oben, sonst steht der Kasten mitten in den Kurven.
    if mehrere:
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi * 1.9 if LOG_Y else hi + 0.28 * (hi - lo))
        ax.legend(loc="upper left", ncol=len(RICHTUNGEN) * len(FAMILIEN),
                  frameon=False, handlelength=1.3, handletextpad=0.5,
                  columnspacing=1.1, borderaxespad=0.4)

for o in fig.findobj(plt.Text):
    o.set_fontsize(PT)
for ax in fig.axes:
    ax.tick_params(labelsize=PT)
fig.tight_layout()

pdf = OUT / f"{NAME}_w{WAIST_UM:.2f}um_width{WIDTH_MHZ:.2f}MHz.pdf"
fig.savefig(pdf)
print("gespeichert:", pdf.name)

gespeichert: AtomOffset_Richtungen_w1.04um_width0.37MHz.pdf


In [4]:
# ====================  DATEN SPEICHERN  ==============================
# Lange Form, weil die Richtungen verschieden weit laufen.
kopf = ["richtung", "r_um", "r_pro_halbe_spannweite"]
for fam, idx, _f, _ls in FAMILIEN:
    kopf += [f"U_{idx}_prozent", f"eta_{idx}_prozent"]
if HART:
    kopf.append("J_prozent")

zeilen = []
for rid in RICHTUNGEN:
    r_um = res[rid]["offset_um"]
    spalten = [r_um, r_um / rand_um]
    for fam, idx, _f, _ls in FAMILIEN:
        spalten += [res[rid][f"uniformity_{fam}__{rid}"] * 100.0,
                    res[rid][f"crosstalk_{fam}__{rid}"] * 100.0]
    if HART:
        spalten.append(res[rid][f"combined_score__{rid}"] * 100.0)
    for werte in zip(*spalten):
        zeilen.append([TEXTE[rid].rstrip(".")] + [f"{v:.6g}" for v in werte])

csv = OUT / f"{NAME}_w{WAIST_UM:.2f}um_width{WIDTH_MHZ:.2f}MHz.csv"
with open(csv, "w", encoding="utf-8") as f:
    f.write(f"# 3x4, waist {WAIST_UM} um, width {WIDTH_MHZ} MHz, r_x {R_X}, "
            f"r_y {R_Y}, {PROFIL}\n"
            f"# Spannweite Tonarray d = {d_um:.4f} um, Rand (d/2) = {rand_um:.4f} um, "
            f"sigma_atom = {opt.sigma_atom*1e9:.1f} nm\n"
            f"# Diagonale laeuft bis zum Eckton, also bis d/2 * sqrt(2)\n")
    f.write(",".join(kopf) + "\n")
    for z in zeilen:
        f.write(",".join(z) + "\n")
print("gespeichert:", csv.name, f"({len(zeilen)} Zeilen)")

# Uebersicht je Richtung und Groesse
print(f"\n{'Richtung':10s} {'Groesse':8s} {'r=0':>8s} {'Maximum':>9s} {'bei r':>8s} "
      f"{'Ende':>9s} {'bei r':>8s}")
for rid in RICHTUNGEN:
    r_um = res[rid]["offset_um"]
    for fam, idx, _f, _ls in FAMILIEN:
        for groesse, sym in (("uniformity", "U_" + idx), ("crosstalk", "eta_" + idx)):
            y = res[rid][f"{groesse}_{fam}__{rid}"] * 100.0
            i = int(np.nanargmax(y))
            print(f"{TEXTE[rid]:10s} {sym:8s} {y[0]:8.3f} {y[i]:9.3f} {r_um[i]:8.3f} "
                  f"{y[-1]:9.3f} {r_um[-1]:8.3f}")

gespeichert: AtomOffset_Richtungen_w1.04um_width0.37MHz.csv (123 Zeilen)

Richtung   Groesse       r=0   Maximum    bei r      Ende    bei r
Hor.       U_w         0.429     3.750    1.169     3.750    1.169
Hor.       eta_w       1.398     3.252    0.701     2.028    1.169
Vert.      U_w         0.429     6.867    1.169     6.867    1.169
Vert.      eta_w       1.398     2.384    0.847     2.321    1.169
Diag.      U_w         0.429     8.988    1.653     8.988    1.653
Diag.      eta_w       1.398     3.514    0.950     2.917    1.653


## Notizen